# Phase 2: Filtering Iranian Titles and Merging Datasets

<div dir='ltr' class='alert alert-info'>In this notebook, we aim to isolate Iranian cinematic works from the massive IMDb dataset. 
Since IMDb does not have a straightforward "Country of Origin" column in its primary tables, we use a heuristic approach:
    
1. Identify titles localized in Iran (region == 'IR') with Persian language (language == 'fa').
2. Filter out global blockbusters (dubbed in Persian) by restricting titles to those with a low global footprint.
3. Merge the filtered titles with the "title.principals" dataset using chunking.</div>

In [1]:
import pandas as pd

In [2]:
df_basics = pd.read_csv('../data/raw/title.basics.tsv.gz', sep='\t', low_memory=False)

In [3]:
df_akas = pd.read_csv('../data/raw/title.akas.tsv.gz', sep='\t', low_memory=False)

<div dir='ltr' class='alert alert-warning'>
Step 1: Identify Initial Candidates
    
Extract titles that have an Iranian regional release and use the Persian language.
</div>

In [4]:
iran_candidates = df_akas[(df_akas['region'] == 'IR') & (df_akas['language'] == 'fa')]['titleId'].unique()

In [5]:
print(len(iran_candidates))

4503


<div dir='ltr' class='alert alert-warning'>
Step 2: Reduce False Positives

Many famous foreign movies are dubbed in Persian and tagged with region == 'IR'. 
To filter these out, we assume that Iranian films generally have a smaller international distribution . We restrict our list to titles released in 5 or fewer regions.
</div>

In [6]:
region_counts = df_akas.groupby('titleId')['region'].nunique()
low_region_count = region_counts[region_counts <= 5].index

In [7]:
iran_titles_v3 = set(iran_candidates) & set(low_region_count) #intersection
print(len(iran_titles_v3))

1854


<div dir='ltr' class='alert alert-warning'>
Step 3: Filter by Title Type
    
Exclude video games, short films, and miscellaneous content.
</div>

In [9]:
iran_basics = df_basics[df_basics['tconst'].isin(iran_titles_v3)]
iran_basics['titleType'].value_counts()

titleType
movie           986
short           463
tvSeries        233
tvMovie          75
video            47
tvMiniSeries     36
videoGame         8
tvSpecial         4
tvShort           2
Name: count, dtype: int64

In [10]:
selected_types = ['movie', 'tvSeries', 'tvMovie', 'tvMiniSeries']
iranian_tconst = df_basics[df_basics['tconst'].isin(iran_titles_v3) & df_basics['titleType'].isin(selected_types)]
iran_tconst_set = set(iranian_tconst['tconst']) 
print("final Irainian set of films: ", len(iran_tconst_set))

final Irainian set of films:  1330


<div dir='ltr' class='alert alert-warning'>
Step 4: Merging with Principals

The "title.principals" file is too large to fit into memory all at once. 
We process it in chunks of 1 million rows, filtering for our targeted Iranian titles, and then merge the results.
</div>

In [11]:
chunks = []
for chunk in pd.read_csv('../data/raw/title.principals.tsv.gz', sep='\t', chunksize=1_000_000, low_memory=False):
    filtered = chunk[chunk['tconst'].isin(iran_tconst_set)]
    chunks.append(filtered)

df_principals_iran = pd.concat(chunks, ignore_index=True)

df_merged = iranian_tconst.merge(df_principals_iran, on='tconst', how='inner')
print(df_merged.shape)
df_merged.head()

(17107, 14)


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,ordering,nconst,category,job,characters
0,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",1,nm0798270,actor,\N,"[""Fernando Torres (Huracán Ramírez)""]"
1,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",2,nm0739131,actress,\N,"[""Margarita""]"
2,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",3,nm0327643,actress,\N,"[""Laura""]"
3,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",4,nm0273508,actor,\N,"[""El pichi""]"
4,tt0044728,movie,Huracán Ramírez,Huracán Ramírez,0,1953,\N,103,"Action,Comedy,Crime",5,nm0414109,actor,\N,"[""Señor Torres (Tonina Jackson)""]"


<div dir='ltr' class='alert alert-warning'>
Step 5: Save Processed Data to use later on

Save the merged dataset in Parquet format.
</div>

In [12]:
# saved df_merged on hard disk to use later 
import os
os.makedirs('../data/processed', exist_ok=True)
df_merged.to_parquet('../data/processed/iran_films_principals.parquet')
print("saved: ", df_merged.shape)

saved:  (17107, 14)


In [13]:
df_akas[df_akas['titleId'] == 'tt0044728']

,titleId,ordering,title,region,language,types,attributes,isOriginalTitle
417152,tt0044728,1,Huracán Ramírez,\N,\N,original,\N,1
417153,tt0044728,2,Huracán Ramírez,BR,\N,imdbDisplay,\N,0
417154,tt0044728,3,Huracán Ramírez,MX,\N,imdbDisplay,\N,0
417155,tt0044728,4,Qahremane shekastnapazir,IR,fa,imdbDisplay,\N,0
